<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/Structured_Outputs_Project_Catalog_Enrichment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 3: Bulk Catalog Enrichment, Async and Batch

Companion notebook for the lesson **Applied Structured Outputs: Three Mini Projects**.

The most common structured output workload in practice is the bulk backfill: N records, one extraction each. We enrich a catalog of course articles with summaries, tags, and a difficulty level, which is exactly the RAG metadata that later powers filtered retrieval.

**Two production patterns:**

1. **Async parallel calls** with a semaphore: low latency, you manage concurrency.
2. **The Batch API**: an asynchronous job at 50% of interactive pricing, best for large backfills where you can wait.

## Install Packages and Set Up the Provider

Pick your provider by setting `PROVIDER` below. Gemini is the course default and its free tier covers this notebook.

> **Colab users:** store your API key in **Secrets** (the key icon in the left sidebar) under the name shown for your provider (`GOOGLE_API_KEY`, `OPENAI_API_KEY`, or `ANTHROPIC_API_KEY`). The cell falls back to an interactive prompt if no secret is found.

In [1]:
# Shared install profile for the structured outputs project notebooks (pin set checked August 2026)
!pip install -q google-genai==2.18.0 openai==3.0.0 anthropic==0.122.0 pydantic==2.13.4 pandas==3.0.5 tqdm==4.70.0

In [2]:
import os
import getpass

PROVIDER = "gemini"  # "gemini" | "openai" | "anthropic"

KEY_ENV = {
    "gemini": "GOOGLE_API_KEY",
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
}
env_var = KEY_ENV[PROVIDER]

# Option 1: Colab Secrets (recommended)
try:
    from google.colab import userdata
    os.environ[env_var] = userdata.get(env_var)
except Exception:
    pass

# Option 2: interactive prompt (fallback)
if not os.getenv(env_var):
    os.environ[env_var] = getpass.getpass(f"Enter {env_var}: ")

print(f"[OK] {env_var} is set")

[OK] GOOGLE_API_KEY is set


### The `extract()` Helper

All project code goes through one function, `extract(prompt, schema)`. It sends a prompt and returns a validated Pydantic object, using the native structured output API of whichever provider you selected. We use each provider's small, fast model: extraction is high-volume, low-difficulty work, and the flagship models cost several times more per token while adding little on tasks this constrained. (Model IDs current as of August 2026. Swap in the provider's latest small model if these have been superseded.)

In [3]:
from pydantic import BaseModel

MODELS = {
    "gemini": "gemini-3.5-flash-lite",
    "openai": "gpt-5.6-luna",
    "anthropic": "claude-haiku-4-5",
}

if PROVIDER == "gemini":
    from google import genai
    client = genai.Client()
elif PROVIDER == "openai":
    from openai import OpenAI
    client = OpenAI()
elif PROVIDER == "anthropic":
    import anthropic
    client = anthropic.Anthropic()

def extract(prompt: str, schema: type[BaseModel], system: str | None = None,
            model: str | None = None) -> BaseModel:
    """Send a prompt and return a validated instance of `schema`."""
    if PROVIDER == "gemini":
        response = client.models.generate_content(
            model=model or MODELS["gemini"],
            contents=prompt,
            config={
                "system_instruction": system,
                "response_mime_type": "application/json",
                "response_schema": schema,
            },
        )
        if response.parsed is None:
            raise ValueError("Model returned no parseable output")
        return response.parsed
    if PROVIDER == "openai":
        response = client.responses.parse(
            model=model or MODELS["openai"],
            instructions=system,
            input=prompt,
            text_format=schema,
            reasoning={"effort": "none"},
        )
        return response.output_parsed
    if PROVIDER == "anthropic":
        response = client.messages.parse(
            model=model or MODELS["anthropic"],
            max_tokens=2048,
            **({"system": system} if system else {}),
            messages=[{"role": "user", "content": prompt}],
            output_format=schema,
        )
        return response.parsed_output
    raise ValueError(f"Unknown provider: {PROVIDER}")

print(f"[OK] extract() ready, provider={PROVIDER}, model={MODELS[PROVIDER]}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


[OK] extract() ready, provider=gemini, model=gemini-3.5-flash-lite


### The `extract_async()` Helper

Part 1 needs the async client. It covers the same three providers and returns the same types, now awaitable.

In [4]:
import asyncio

if PROVIDER == "gemini":
    async_client = client.aio
elif PROVIDER == "openai":
    from openai import AsyncOpenAI
    async_client = AsyncOpenAI()
elif PROVIDER == "anthropic":
    import anthropic
    async_client = anthropic.AsyncAnthropic()

async def extract_async(prompt: str, schema: type[BaseModel], system: str | None = None,
                        model: str | None = None) -> BaseModel:
    """Async version of extract()."""
    if PROVIDER == "gemini":
        response = await async_client.models.generate_content(
            model=model or MODELS["gemini"],
            contents=prompt,
            config={
                "system_instruction": system,
                "response_mime_type": "application/json",
                "response_schema": schema,
            },
        )
        if response.parsed is None:
            raise ValueError("Model returned no parseable output")
        return response.parsed
    if PROVIDER == "openai":
        response = await async_client.responses.parse(
            model=model or MODELS["openai"],
            instructions=system,
            input=prompt,
            text_format=schema,
            reasoning={"effort": "none"},
        )
        return response.output_parsed
    if PROVIDER == "anthropic":
        response = await async_client.messages.parse(
            model=model or MODELS["anthropic"],
            max_tokens=2048,
            **({"system": system} if system else {}),
            messages=[{"role": "user", "content": prompt}],
            output_format=schema,
        )
        return response.parsed_output
    raise ValueError(f"Unknown provider: {PROVIDER}")

print("[OK] extract_async() ready")

[OK] extract_async() ready


## The Catalog

This is a small synthetic catalog of course articles. In your own projects, this would be a CSV export of whatever needs enriching: products, documents, chunks.

In [5]:
records = [
    {"id": "A001", "title": "A Gentle Introduction to Transformers", "author": "E. Rivera",
     "year": 2023, "blurb": "How attention replaced recurrence, with a minimal PyTorch implementation."},
    {"id": "A002", "title": "Build Your First RAG Pipeline", "author": "M. Thompson",
     "year": 2024, "blurb": "Chunking, embeddings, and retrieval from scratch in plain Python."},
    {"id": "A003", "title": "Vector Databases Compared", "author": "S. Patel",
     "year": 2025, "blurb": "Chroma, Qdrant, and pgvector benchmarked on a documentation corpus."},
    {"id": "A004", "title": "Fine-Tuning Small Models on a Budget", "author": "A. Morales",
     "year": 2024, "blurb": "LoRA and QLoRA walkthroughs that run on a single consumer GPU."},
    {"id": "A005", "title": "Evaluating LLM Applications", "author": "J. Kim",
     "year": 2025, "blurb": "Hit rate, MRR, and LLM judges: how to measure whether your pipeline works."},
    {"id": "A006", "title": "Agents and Tool Calling in Production", "author": "L. Garcia",
     "year": 2026, "blurb": "Designing tool schemas, handling failures, and keeping agents on task."},
    {"id": "A007", "title": "The Economics of LLM APIs", "author": "T. Nguyen",
     "year": 2026, "blurb": "Token pricing, caching, and batch discounts, with worked cost examples."},
    {"id": "A008", "title": "Prompt Engineering Patterns That Survive Model Upgrades", "author": "N. Chen",
     "year": 2025, "blurb": "Practical prompting techniques and how to keep them working as models change."},
    {"id": "A009", "title": "From Notebook to Production API", "author": "R. Singh",
     "year": 2025, "blurb": "Turning an LLM prototype into a deployed FastAPI service with monitoring."},
    {"id": "A010", "title": "Understanding Tokenizers", "author": "K. Johnson",
     "year": 2024, "blurb": "Why models see subwords, and what that means for counting, cost, and multilingual text."},
]

print(f"Loaded {len(records)} catalog records")

Loaded 10 catalog records


## The Enrichment Schema and Prompt

Note what the prompt does not contain: a list of output fields. With native schema enforcement, the schema already carries the field names, types, and descriptions. Repeating them in the prompt creates a second copy that can drift, so keep the schema as the single source of truth.

In [6]:
from pydantic import Field
from typing import List, Literal

class EnrichedRecord(BaseModel):
    summary: str = Field(description="1-2 sentence plain-language summary for the catalog page")
    tags: List[str] = Field(description="3-8 lowercase topical tags")
    difficulty: Literal["beginner", "intermediate", "advanced"] = Field(
        description="Level of prior ML/LLM knowledge the article assumes"
    )
    confidence: float = Field(ge=0, le=1, description="Self-reported confidence, 0-1")

def build_prompt(row: dict) -> str:
    return f"""Enrich this course-catalog record. Base the output only on the provided fields.

ID: {row['id']}
Title: {row['title']}
Author: {row['author']}
Year: {row['year']}
Blurb: {row['blurb']}"""

# Quick smoke test with the synchronous helper
item = extract(build_prompt(records[0]), EnrichedRecord)
print(f"{records[0]['id']}: [{item.difficulty}] {', '.join(item.tags[:4])}")
print(item.summary)

A001: [beginner] transformers, attention, pytorch, deep learning
This course provides a gentle introduction to transformers by explaining how attention replaced recurrence, complete with a minimal PyTorch implementation.


## Part 1: Concurrent Calls with a Semaphore

A sequential loop over 10,000 records at two seconds per call takes over five hours. The calls are independent, so we issue them concurrently. But unbounded concurrency hits rate limits and starts failing with 429 errors, so a semaphore caps how many calls run at once.

`return_exceptions=True` plays the role a try/except plays in a sequential loop: one failed record surfaces as a result instead of killing the whole gather. The concurrency cap is a dial: keep it at 2 to 4 on a free tier, raise it on a paid tier.

In [7]:
async def enrich_all(rows: list[dict], concurrency: int = 4) -> list:
    semaphore = asyncio.Semaphore(concurrency)

    async def enrich_one(row: dict):
        async with semaphore:
            return await extract_async(build_prompt(row), EnrichedRecord)

    return await asyncio.gather(*[enrich_one(r) for r in rows], return_exceptions=True)

enriched = await enrich_all(records, concurrency=4)

for row, item in zip(records, enriched):
    if isinstance(item, Exception):
        print(f"{row['id']}: FAILED ({item})")
    else:
        print(f"{row['id']}: [{item.difficulty}] {', '.join(item.tags[:4])}")

A001: [beginner] transformers, attention, pytorch, deep learning
A002: [beginner] rag, python, embeddings, retrieval
A003: [intermediate] vector databases, chroma, qdrant, pgvector
A004: [intermediate] fine-tuning, lora, qlora, gpu
A005: [intermediate] evaluation, llm, metrics, pipelines
A006: [intermediate] agents, tool calling, production, schemas
A007: [beginner] economics, apis, llm, pricing
A008: [intermediate] prompt engineering, model upgrades, robustness, llm
A009: [intermediate] llm, fastapi, deployment, production
A010: [beginner] tokenization, subwords, llm, cost optimization


> **Try it:** rerun the cell with `concurrency=1` and `concurrency=8` and time both (`%%time` at the top of the cell). Watch for 429 errors at the high end. That is your tier's rate limit talking.

## Part 2: The Batch API at Half Price

Async calls optimize for latency, which a backfill usually does not need. A batch API trades that latency away: you submit the whole job up front and the provider processes it on spare capacity within a day. Batch work costs 50% of the interactive price. All three providers offer this:

- [Gemini Batch API](https://ai.google.dev/gemini-api/docs/batch-api)
- [OpenAI Batch API](https://platform.openai.com/docs/guides/batch)
- [Anthropic Message Batches](https://platform.claude.com/docs/en/build-with-claude/batch-processing)

Run the subsection that matches your `PROVIDER`. Batch jobs are asynchronous: they can finish in minutes or take hours, so expect to leave the polling cell running or come back later. Batch processing generally requires a paid-tier API key.

Two rules apply everywhere: give every request an ID and match results by that ID, never by position. Re-validate every result with Pydantic on the way in, since batch results come back as plain text.

### Gemini

Four steps: write a JSONL file with one request per line, upload it, create the job, poll and download. The `key` field is our record ID, and `responseJsonSchema` keeps batch outputs schema-enforced exactly like interactive ones.

In [8]:
import json

jsonl_path = "catalog_requests.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for row in records:
        line = {
            "key": row["id"],
            "request": {
                "contents": [{"role": "user", "parts": [{"text": build_prompt(row)}]}],
                "generationConfig": {
                    "responseMimeType": "application/json",
                    "responseJsonSchema": EnrichedRecord.model_json_schema(),
                },
            },
        }
        f.write(json.dumps(line) + "\n")

uploaded = client.files.upload(file=jsonl_path, config={"mime_type": "application/jsonl"})
job = client.batches.create(model=MODELS["gemini"], src=uploaded.name,
                            config={"display_name": "catalog-enrichment"})
print("Created:", job.name, "state:", job.state.name)

Created: batches/pqsdquz234vtfvizb8zr66rt0l40ual5565x state: JOB_STATE_PENDING


In [9]:
import time

DONE = {"JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED"}

while job.state.name not in DONE:
    print("state:", job.state.name, "- waiting 30s")
    time.sleep(30)
    job = client.batches.get(name=job.name)

print("final state:", job.state.name)

state: JOB_STATE_PENDING - waiting 30s


state: JOB_STATE_RUNNING - waiting 30s


state: JOB_STATE_RUNNING - waiting 30s


state: JOB_STATE_RUNNING - waiting 30s


final state: JOB_STATE_SUCCEEDED


In [10]:
if job.state.name == "JOB_STATE_SUCCEEDED":
    content = client.files.download(file=job.dest.file_name).decode("utf-8")
    enriched_by_id = {}
    for line in content.splitlines():
        record = json.loads(line)
        text = record["response"]["candidates"][0]["content"]["parts"][0]["text"]
        enriched_by_id[record["key"]] = EnrichedRecord.model_validate_json(text)
    print(f"Parsed {len(enriched_by_id)} records")
    for rid, item in list(enriched_by_id.items())[:3]:
        print(f"{rid}: [{item.difficulty}] {item.summary}")
elif job.state.name == "JOB_STATE_FAILED":
    print("Job failed:", job.error)

Parsed 10 records
A001: [beginner] An introductory guide explaining how attention mechanisms replaced recurrence in machine learning, complete with a minimal PyTorch implementation.
A002: [beginner] Learn to build a Retrieval-Augmented Generation pipeline from scratch using plain Python, covering chunking, embeddings, and retrieval.
A003: [intermediate] A comparative benchmark of Chroma, Qdrant, and pgvector using a standard documentation corpus.


### OpenAI

Same JSONL shape, and each line names the endpoint to call. The `text.format` block carries the JSON schema. (Check the [Batch API docs](https://platform.openai.com/docs/guides/batch) for the current request-body shape before running a large job.)

In [ ]:
import json
import time

def openai_request_line(row: dict) -> dict:
    return {
        "custom_id": row["id"],
        "method": "POST",
        "url": "/v1/responses",
        "body": {
            "model": MODELS["openai"],
            "input": build_prompt(row),
            "text": {
                "format": {
                    "type": "json_schema",
                    "name": "enriched_record",
                    "strict": True,
                    "schema": {**EnrichedRecord.model_json_schema(), "additionalProperties": False},
                }
            },
        },
    }

with open("openai_requests.jsonl", "w", encoding="utf-8") as f:
    for row in records:
        f.write(json.dumps(openai_request_line(row)) + "\n")

batch_file = client.files.create(file=open("openai_requests.jsonl", "rb"), purpose="batch")
batch = client.batches.create(input_file_id=batch_file.id,
                              endpoint="/v1/responses", completion_window="24h")
print("Created:", batch.id, batch.status)

In [ ]:
while True:
    batch = client.batches.retrieve(batch.id)
    if batch.status in {"completed", "failed", "expired", "cancelled"}:
        break
    print("status:", batch.status, "- waiting 60s")
    time.sleep(60)

if batch.status == "completed":
    enriched_by_id = {}
    for line in client.files.content(batch.output_file_id).text.splitlines():
        rec = json.loads(line)
        body = rec["response"]["body"]
        text = next(
            part["text"]
            for item in body["output"] if item["type"] == "message"
            for part in item["content"] if part["type"] == "output_text"
        )
        enriched_by_id[rec["custom_id"]] = EnrichedRecord.model_validate_json(text)
    print(f"Parsed {len(enriched_by_id)} records")
else:
    print("Batch ended with status:", batch.status)

### Anthropic

No file step: pass the request list directly, then stream results keyed by `custom_id`. Most batches finish within an hour, with a 24-hour ceiling.

In [ ]:
import time
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

STRICT_SCHEMA = {**EnrichedRecord.model_json_schema(), "additionalProperties": False}

batch = client.messages.batches.create(
    requests=[
        Request(
            custom_id=row["id"],
            params=MessageCreateParamsNonStreaming(
                model=MODELS["anthropic"],
                max_tokens=1024,
                messages=[{"role": "user", "content": build_prompt(row)}],
                output_config={"format": {"type": "json_schema", "schema": STRICT_SCHEMA}},
            ),
        )
        for row in records
    ]
)
print("Created:", batch.id, batch.processing_status)

In [ ]:
while True:
    batch = client.messages.batches.retrieve(batch.id)
    if batch.processing_status == "ended":
        break
    print("status:", batch.processing_status, "- waiting 60s")
    time.sleep(60)

enriched_by_id = {}
for result in client.messages.batches.results(batch.id):
    if result.result.type == "succeeded":
        msg = result.result.message
        text = next((b.text for b in msg.content if b.type == "text"), "")
        enriched_by_id[result.custom_id] = EnrichedRecord.model_validate_json(text)
    else:
        print(f"[{result.custom_id}] {result.result.type}")

print(f"Parsed {len(enriched_by_id)} records")

## Choosing a Pattern

| Pattern | Latency | Cost | Use it for |
| --- | --- | --- | --- |
| Sequential loop | Minutes for hundreds of records | Full price | Small jobs, debugging, notebooks |
| Async + semaphore | Seconds to minutes | Full price | User-facing jobs, moderate scale |
| Batch API | Up to 24 hours | 50% price | Backfills, nightly jobs, eval runs |

## Exercises

1. **Cache what you paid for.** Persist `enriched_by_id` to a JSON file and make the async runner skip IDs that already have results. Rerun and confirm zero API calls.
2. **Retries with backoff.** Wrap `extract_async()` with up to 3 retries and exponential backoff on rate-limit errors, then push `concurrency` up until the retries start firing.
3. **Ship it.** Write the enriched catalog to a CSV with one column per schema field. This is the metadata a vector database filters on, which the metadata-filtering lesson later in the course puts to work.
4. **Price it.** Using your provider's current pricing page, estimate the cost of enriching 100,000 records interactively vs through the batch API.